# Gold Layer — Dimensiuni îmbogățite

**Scop:** Construirea celor 3 dimensiuni îmbogățite din Gold:
1. **`dim_product`** — catalog de produse bancare (combinație din ref_* + atribute marketing)
2. **`dim_customer_enriched`** — denormalizare clienți + agregate per client
3. **`dim_risk_profile`** — profil de risc calculat per client (scoruri normalizate)

**Diferența față de Silver:**
- Silver = model dimensional la granularitate originală (un rând per entitate)
- Gold = dimensiuni **îmbogățite** prin denormalizare și calcule agregate (mai utile pentru consum)

**Strategie:** `overwrite` la fiecare rulare — dimensiunile Gold se recalculează din Silver.

## 1. Configurare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    count, sum as f_sum, avg, max as f_max, min as f_min,
    countDistinct, datediff, current_date, year, month,
    coalesce, expr, round as f_round, floor,
    least, greatest, percent_rank,
    months_between, datediff as f_datediff,
)
from pyspark.sql.types import IntegerType, DoubleType, StringType
from pyspark.sql.window import Window

CATALOG = "banking"
SILVER  = "silver"
GOLD    = "gold"

print(f"Silver in:  {CATALOG}.{SILVER}.*")
print(f"Gold out:   {CATALOG}.{GOLD}.*")

# Asiguram schema gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")

Silver in:  banking.silver.*
Gold out:   banking.gold.*


DataFrame[]

## 2. `dim_product` — catalog produse bancare îmbogățit

**Sursa:** combinăm `ref_account_types`, `ref_card_types`, `ref_loan_types` din Silver.

**Atribute adăugate (marketing/business):**
- `product_id` — surrogate key generat (PROD-XXX)
- `category` — ACCOUNT / CARD / LOAN
- `commercial_name` — nume marketing pentru UI
- `target_segment` — pentru ce segment de clienți e produsul
- `is_active_for_sale` — disponibil pentru vânzări noi (vs. legacy)
- `priority` — ordine în catalog (1=top, 10=jos)

In [0]:
print("Construire dim_product...")

# ── Account types ─────────────────────────────────────
acc_types = spark.table(f"{CATALOG}.{SILVER}.ref_account_types").select(
    col("type_code").alias("source_code"),
    col("name").alias("technical_name"),
    col("interest_rate_pa"),
    col("min_balance"),
)

# Adaugam atribute Gold
acc_types = (acc_types
    .withColumn("category",          lit("ACCOUNT"))
    .withColumn("commercial_name",
        when(col("source_code") == "CURRENT",  lit("Cont Curent Standard"))
        .when(col("source_code") == "SAVINGS", lit("Cont de Economii Plus"))
        .when(col("source_code") == "DEPOSIT", lit("Depozit la Termen"))
        .when(col("source_code") == "LOAN_ACC",lit("Cont de Credit"))
        .otherwise(col("technical_name")))
    .withColumn("target_segment",
        when(col("source_code") == "DEPOSIT", lit("PREMIUM"))
        .when(col("source_code") == "LOAN_ACC", lit("ALL"))
        .otherwise(lit("RETAIL")))
    .withColumn("is_active_for_sale", lit(True))
    .withColumn("annual_fee",         lit(0.0))
    .withColumn("interest_rate",      col("interest_rate_pa"))
)

# ── Card types ────────────────────────────────────────
card_types = spark.table(f"{CATALOG}.{SILVER}.ref_card_types").select(
    col("card_type_code").alias("source_code"),
    col("name").alias("technical_name"),
    col("annual_fee"),
    col("network"),
    col("has_credit_line"),
)

card_types = (card_types
    .withColumn("category",          lit("CARD"))
    .withColumn("commercial_name",
        when(col("source_code") == "DEBIT_VISA",   lit("Card Visa Debit"))
        .when(col("source_code") == "DEBIT_MC",    lit("Card Mastercard Debit"))
        .when(col("source_code") == "CREDIT_VISA", lit("Card Visa Credit Gold"))
        .when(col("source_code") == "CREDIT_MC",   lit("Card Mastercard Credit Gold"))
        .when(col("source_code") == "PREPAID_VISA",lit("Card Prepaid Visa"))
        .when(col("source_code") == "VIRTUAL_MC",  lit("Card Virtual Mastercard"))
        .otherwise(col("technical_name")))
    .withColumn("target_segment",
        when(col("source_code").like("CREDIT_%"), lit("PREMIUM"))
        .when(col("source_code") == "VIRTUAL_MC", lit("RETAIL"))
        .otherwise(lit("RETAIL")))
    .withColumn("is_active_for_sale", lit(True))
    .withColumn("interest_rate",      lit(None).cast(DoubleType()))
    .withColumn("min_balance",        lit(None).cast(DoubleType()))
    .drop("network", "has_credit_line")
)

# ── Loan types ────────────────────────────────────────
loan_types = spark.table(f"{CATALOG}.{SILVER}.ref_loan_types").select(
    col("loan_type_code").alias("source_code"),
    col("name").alias("technical_name"),
    col("typical_rate_pa").alias("interest_rate"),
)

loan_types = (loan_types
    .withColumn("category",          lit("LOAN"))
    .withColumn("commercial_name",
        when(col("source_code") == "MORTGAGE",    lit("Credit Ipotecar Casa Ta"))
        .when(col("source_code") == "PERSONAL",   lit("Credit Personal Rapid"))
        .when(col("source_code") == "AUTO",       lit("Credit Auto"))
        .when(col("source_code") == "OVERDRAFT",  lit("Linie de Overdraft"))
        .when(col("source_code") == "SME_WORKING",lit("Credit Capital de Lucru IMM"))
        .when(col("source_code") == "SME_INVEST", lit("Credit Investitii IMM"))
        .otherwise(col("technical_name")))
    .withColumn("target_segment",
        when(col("source_code").like("SME_%"),    lit("SME"))
        .when(col("source_code") == "MORTGAGE",   lit("ALL"))
        .otherwise(lit("RETAIL")))
    .withColumn("is_active_for_sale", lit(True))
    .withColumn("annual_fee",         lit(0.0))
    .withColumn("min_balance",        lit(None).cast(DoubleType()))
)

# ── UNION cele 3 categorii ────────────────────────────
common_cols = [
    "source_code", "category", "commercial_name", "technical_name",
    "target_segment", "is_active_for_sale",
    "annual_fee", "interest_rate", "min_balance"
]

dim_product = (
    acc_types.select(*common_cols)
    .unionByName(card_types.select(*common_cols))
    .unionByName(loan_types.select(*common_cols))
)

# ── Adaugare product_id si priority ──────────────────
window = Window.orderBy("category", "source_code")
dim_product = (dim_product
    .withColumn("_rn", expr("row_number() OVER (ORDER BY category, source_code)"))
    .withColumn("product_id", expr("concat('PROD-', lpad(cast(_rn as string), 3, '0'))"))
    .withColumn("priority", col("_rn"))
    .drop("_rn")
)

# Reordonam coloanele
dim_product = dim_product.select(
    "product_id", "category", "source_code",
    "commercial_name", "technical_name",
    "target_segment", "is_active_for_sale", "priority",
    "annual_fee", "interest_rate", "min_balance"
).withColumn("gold_loaded_at", current_timestamp())

# Scriem in Gold
(dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_product"))

print(f"  dim_product: {dim_product.count()} produse")
dim_product.show(20, truncate=False)

Construire dim_product...
  dim_product: 16 produse
+----------+--------+------------+---------------------------+---------------------------+--------------+------------------+--------+----------+-------------+-----------+--------------------------+
|product_id|category|source_code |commercial_name            |technical_name             |target_segment|is_active_for_sale|priority|annual_fee|interest_rate|min_balance|gold_loaded_at            |
+----------+--------+------------+---------------------------+---------------------------+--------------+------------------+--------+----------+-------------+-----------+--------------------------+
|PROD-001  |ACCOUNT |CURRENT     |Cont Curent Standard       |Cont curent                |RETAIL        |true              |1       |0.0       |0.0          |0.0        |2026-05-02 22:10:26.024778|
|PROD-002  |ACCOUNT |DEPOSIT     |Depozit la Termen          |Depozit la termen          |PREMIUM       |true              |2       |0.0       |5.25        

## 3. `dim_customer_enriched` — clienți denormalizați + agregate

**Sursa:** `dim_customers` (versiunea curentă SCD2) + agregate din `dim_accounts`, `dim_cards`, `dim_loans`, `fact_transactions`.

**Atribute adăugate (calculate):**
- `age` — vârsta calculată din `date_of_birth`
- `age_group` — grupa de vârstă (18-25, 26-35, 36-50, 51-65, 66+)
- `tenure_years` — câți ani e client (din primul cont deschis)
- `total_accounts`, `total_cards`, `total_loans` — câte produse are
- `has_loan`, `has_card`, `has_savings` — flag-uri rapide
- `total_balance_ron` — sold total pe toate conturile (echivalent RON)
- `lifetime_volume_ron` — volum total tranzacționat
- `lifetime_tx_count` — număr total tranzacții
- `last_tx_date` — ultima activitate

In [0]:
print("Construire dim_customer_enriched...")

# Pornim de la versiunea curenta SCD2 a clientilor
df_cust = spark.table(f"{CATALOG}.{SILVER}.dim_customers").filter(col("is_current") == True)

# ── Calcul varsta + grupa ─────────────────────────────
df_cust = (df_cust
    .withColumn("age", floor(months_between(current_date(), col("date_of_birth")) / 12).cast(IntegerType()))
    .withColumn("age_group",
        when(col("age") < 26, lit("18-25"))
        .when(col("age") < 36, lit("26-35"))
        .when(col("age") < 51, lit("36-50"))
        .when(col("age") < 66, lit("51-65"))
        .otherwise(lit("66+")))
)

# ── Agregate per client din dim_accounts ──────────────
acc_agg = (spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
    .filter(col("status") == "ACTIVE")
    .groupBy("customer_id")
    .agg(
        count("account_id").alias("total_accounts"),
        f_min("open_date").alias("first_account_date"),
        f_sum("balance").alias("total_balance_lcy"),    # local currency, mixed
    )
)

# ── Agregate per client din dim_cards ─────────────────
# Trecem prin dim_accounts pentru a obtine customer_id
cards_with_cust = (spark.table(f"{CATALOG}.{SILVER}.dim_cards")
    .filter(col("status").isin(["ACTIVE", "BLOCKED"]))
    .alias("c")
    .join(
        spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "customer_id").alias("a"),
        col("c.account_id") == col("a.account_id"),
        "inner"
    )
    .select("a.customer_id", "c.card_id", "c.card_type_code")
)

cards_agg = cards_with_cust.groupBy("customer_id").agg(
    count("card_id").alias("total_cards"),
    f_sum(when(col("card_type_code").like("CREDIT_%"), 1).otherwise(0)).alias("credit_cards_count"),
)

# ── Agregate per client din dim_loans ─────────────────
loans_with_cust = (spark.table(f"{CATALOG}.{SILVER}.dim_loans")
    .filter(col("status").isin(["ACTIVE", "OVERDUE"]))
    .alias("l")
    .join(
        spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "customer_id").alias("a"),
        col("l.account_id") == col("a.account_id"),
        "inner"
    )
    .select("a.customer_id", "l.loan_id", "l.outstanding_balance")
)

loans_agg = loans_with_cust.groupBy("customer_id").agg(
    count("loan_id").alias("total_loans"),
    f_sum("outstanding_balance").alias("total_loan_balance"),
)

# ── Agregate din fact_transactions (lifetime) ─────────
tx_agg = (spark.table(f"{CATALOG}.{SILVER}.fact_transactions")
    .filter(col("status_code") == "COMPLETED")
    .groupBy("customer_id")
    .agg(
        count("transaction_id").alias("lifetime_tx_count"),
        f_sum("amount_ron").alias("lifetime_volume_ron"),
        f_max("initiated_at").alias("last_tx_date"),
    )
)

# ── Flag-uri savings ──────────────────────────────────
savings_flag = (spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
    .filter((col("status") == "ACTIVE") & (col("account_type").isin(["SAVINGS", "DEPOSIT"])))
    .select("customer_id")
    .distinct()
    .withColumn("has_savings_calc", lit(True))
)

# ── JOIN totul cu dim_customers (LEFT pentru a pastra clientii fara produse) ──
dim_customer_enriched = (df_cust
    .join(acc_agg,      "customer_id", "left")
    .join(cards_agg,    "customer_id", "left")
    .join(loans_agg,    "customer_id", "left")
    .join(tx_agg,       "customer_id", "left")
    .join(savings_flag, "customer_id", "left")
)

# ── Calcule finale ────────────────────────────────────
dim_customer_enriched = (dim_customer_enriched
    # Default 0 pentru NULL-uri (clienti fara produse)
    .withColumn("total_accounts",       coalesce(col("total_accounts"),       lit(0)))
    .withColumn("total_cards",          coalesce(col("total_cards"),          lit(0)))
    .withColumn("total_loans",          coalesce(col("total_loans"),          lit(0)))
    .withColumn("credit_cards_count",   coalesce(col("credit_cards_count"),   lit(0)))
    .withColumn("lifetime_tx_count",    coalesce(col("lifetime_tx_count"),    lit(0)))
    .withColumn("lifetime_volume_ron",  coalesce(col("lifetime_volume_ron"),  lit(0.0)))
    .withColumn("total_balance_lcy",    coalesce(col("total_balance_lcy"),    lit(0.0)))
    .withColumn("total_loan_balance",   coalesce(col("total_loan_balance"),   lit(0.0)))
    
    # Flag-uri derivate
    .withColumn("has_loan",        col("total_loans") > 0)
    .withColumn("has_card",        col("total_cards") > 0)
    .withColumn("has_credit_card", col("credit_cards_count") > 0)
    .withColumn("has_savings",     coalesce(col("has_savings_calc"), lit(False)))
    
    # Total produse (pentru rapoarte cross-sell)
    .withColumn("total_products", col("total_accounts") + col("total_cards") + col("total_loans"))
    
    # Tenure in ani (din primul cont deschis)
    .withColumn("tenure_years",
        when(col("first_account_date").isNotNull(),
             floor(months_between(current_date(), col("first_account_date")) / 12).cast(IntegerType())
        ).otherwise(lit(0)))
    
    # Lifetime value: combinatie de balanta + volum (formula simpla)
    .withColumn("lifetime_value_ron",
        f_round(col("total_balance_lcy") + col("lifetime_volume_ron") * 0.01, 2))
    
    .drop("has_savings_calc")
    .withColumn("gold_loaded_at", current_timestamp())
)

# Selectam coloanele finale (eliminam metadata SCD2 nerelevanta)
final_cols = [
    "customer_id", "customer_sk",
    "first_name", "last_name",
    "age", "age_group", "gender",
    "county_code", "city",
    "segment_code", "kyc_status", "is_active",
    "tenure_years",
    "total_accounts", "total_cards", "credit_cards_count", "total_loans", "total_products",
    "has_loan", "has_card", "has_credit_card", "has_savings",
    "total_balance_lcy", "total_loan_balance",
    "lifetime_tx_count", "lifetime_volume_ron", "lifetime_value_ron",
    "last_tx_date",
    "gold_loaded_at",
]

dim_customer_enriched = dim_customer_enriched.select(*final_cols)

# Scriem in Gold
(dim_customer_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_customer_enriched"))

print(f"  dim_customer_enriched: {dim_customer_enriched.count()} clienti")

# Sample
print("\n  Sample (top 5 dupa lifetime_value):")
spark.table(f"{CATALOG}.{GOLD}.dim_customer_enriched") \
    .select("customer_id", "segment_code", "age_group", "total_products", 
            "lifetime_value_ron", "tenure_years") \
    .orderBy(col("lifetime_value_ron").desc()) \
    .show(5, truncate=False)

Construire dim_customer_enriched...
  dim_customer_enriched: 500 clienti

  Sample (top 5 dupa lifetime_value):
+-----------+------------+---------+--------------+------------------+------------+
|customer_id|segment_code|age_group|total_products|lifetime_value_ron|tenure_years|
+-----------+------------+---------+--------------+------------------+------------+
|CUST-000274|RETAIL      |36-50    |10            |159248.04         |7           |
|CUST-000213|RETAIL      |66+      |8             |154308.27         |6           |
|CUST-000311|RETAIL      |36-50    |5             |147783.89         |7           |
|CUST-000234|RETAIL      |36-50    |9             |140429.22         |3           |
|CUST-000108|RETAIL      |51-65    |6             |131867.75         |7           |
+-----------+------------+---------+--------------+------------------+------------+
only showing top 5 rows


## 4. `dim_risk_profile` — profil de risc per client

**Calculăm 6 indicatori de risc, fiecare normalizat 0-100:**
1. **`debt_ratio_score`** — gradul de îndatorare (total_loan_balance / lifetime_volume)
2. **`overdraft_score`** — frecvența descoperirilor de cont (% tranzacții FAILED)
3. **`late_payment_score`** — credite cu status OVERDUE
4. **`activity_score`** — inactivitate recentă (zile de la ultima tranzacție)
5. **`balance_volatility_score`** — variabilitate sume tranzacționate
6. **`kyc_score`** — status KYC (FLAGGED/BLOCKED → risc mai mare)

**Scor agregat:** medie ponderată a celor 6 indicatori → mapare în benzi `LOW/MEDIUM/HIGH/CRITICAL`.

In [0]:
print("Construire dim_risk_profile...")

# ── Pornim de la dim_customer_enriched (deja are agregate de baza) ──
df = spark.table(f"{CATALOG}.{GOLD}.dim_customer_enriched")

# ── Indicator 1: Debt ratio score ─────────────────────
# Daca total_loan_balance > lifetime_volume → indatorare mare
df = df.withColumn("debt_ratio_raw",
    when(col("lifetime_volume_ron") > 0,
         col("total_loan_balance") / col("lifetime_volume_ron"))
    .otherwise(lit(0.0))
)
# Normalizare 0-100 (cap la 2.0 ratio = 100)
df = df.withColumn("debt_ratio_score",
    least(lit(100.0), col("debt_ratio_raw") * 50.0)
)

# ── Indicator 2: Overdraft score ──────────────────────
# Procentul de tranzactii FAILED in ultimele 90 zile
tx_recent = (spark.table(f"{CATALOG}.{SILVER}.fact_transactions")
    .filter(datediff(current_date(), col("initiated_at")) <= 90)
    .groupBy("customer_id")
    .agg(
        count("transaction_id").alias("tx_count_90d"),
        f_sum(when(col("status_code") == "FAILED", 1).otherwise(0)).alias("failed_tx_90d"),
    )
)

df = df.join(tx_recent, "customer_id", "left")
df = df.withColumn("overdraft_score",
    when(col("tx_count_90d") > 0,
         (col("failed_tx_90d") / col("tx_count_90d")) * 100.0)
    .otherwise(lit(0.0))
)

# ── Indicator 3: Late payment score ───────────────────
# Numarul de credite OVERDUE per client
overdue_loans = (spark.table(f"{CATALOG}.{SILVER}.dim_loans")
    .filter(col("status") == "OVERDUE")
    .alias("l")
    .join(
        spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "customer_id").alias("a"),
        col("l.account_id") == col("a.account_id"),
        "inner"
    )
    .groupBy("a.customer_id")
    .agg(count("l.loan_id").alias("overdue_loans_count"))
)

df = df.join(overdue_loans, "customer_id", "left")
df = df.withColumn("overdue_loans_count", coalesce(col("overdue_loans_count"), lit(0)))
df = df.withColumn("late_payment_score",
    least(lit(100.0), col("overdue_loans_count") * 33.0)   # 1 overdue=33, 2=66, 3+=100
)

# ── Indicator 4: Activity score ───────────────────────
# Cate zile au trecut de la ultima tranzactie (mai mult = inactivitate = risc)
df = df.withColumn("days_since_last_tx",
    when(col("last_tx_date").isNotNull(),
         datediff(current_date(), col("last_tx_date")))
    .otherwise(lit(365))   # default daca nu exista tranzactii
)
df = df.withColumn("activity_score",
    least(lit(100.0), col("days_since_last_tx") / 3.65)   # 365 zile = 100
)

# ── Indicator 5: Balance volatility score ─────────────
# Coeficient de variatie pe sume in ultimele 90 zile
tx_volatility = (spark.table(f"{CATALOG}.{SILVER}.fact_transactions")
    .filter(
        (datediff(current_date(), col("initiated_at")) <= 90) &
        (col("status_code") == "COMPLETED")
    )
    .groupBy("customer_id")
    .agg(
        avg("amount_ron").alias("avg_amount_90d"),
        expr("stddev(amount_ron)").alias("std_amount_90d"),
    )
)

df = df.join(tx_volatility, "customer_id", "left")
df = df.withColumn("volatility_raw",
    when((col("avg_amount_90d").isNotNull()) & (col("avg_amount_90d") > 0),
         col("std_amount_90d") / col("avg_amount_90d"))
    .otherwise(lit(0.0))
)
df = df.withColumn("balance_volatility_score",
    least(lit(100.0), col("volatility_raw") * 30.0)   # CV de 3.33 = 100
)

# ── Indicator 6: KYC score ────────────────────────────
df = df.withColumn("kyc_score",
    when(col("kyc_status") == "BLOCKED",  lit(100.0))
    .when(col("kyc_status") == "FLAGGED", lit(70.0))
    .when(col("kyc_status") == "PENDING", lit(30.0))
    .when(col("kyc_status") == "VERIFIED",lit(10.0))
    .otherwise(lit(50.0))
)

# ── Scor agregat (medie ponderata) ────────────────────
df = df.withColumn("risk_score",
    f_round(
        col("debt_ratio_score")          * 0.25 +
        col("overdraft_score")           * 0.15 +
        col("late_payment_score")        * 0.25 +
        col("activity_score")            * 0.10 +
        col("balance_volatility_score")  * 0.10 +
        col("kyc_score")                 * 0.15,
        2
    )
)

# ── Mapare in benzi LOW/MEDIUM/HIGH/CRITICAL ──────────
df = df.withColumn("risk_band",
    when(col("risk_score") <= 25, lit("LOW"))
    .when(col("risk_score") <= 60, lit("MEDIUM"))
    .when(col("risk_score") <= 85, lit("HIGH"))
    .otherwise(lit("CRITICAL"))
)

# ── Selectare coloane finale ──────────────────────────
dim_risk_profile = df.select(
    "customer_id", "customer_sk", "segment_code",
    f_round("debt_ratio_score", 2).alias("debt_ratio_score"),
    f_round("overdraft_score", 2).alias("overdraft_score"),
    f_round("late_payment_score", 2).alias("late_payment_score"),
    f_round("activity_score", 2).alias("activity_score"),
    f_round("balance_volatility_score", 2).alias("balance_volatility_score"),
    f_round("kyc_score", 2).alias("kyc_score"),
    "risk_score",
    "risk_band",
    # Date de input pentru transparenta
    "total_loan_balance", "overdue_loans_count",
    "tx_count_90d", "failed_tx_90d",
    "days_since_last_tx",
).withColumn("gold_loaded_at", current_timestamp())

# Scriem in Gold
(dim_risk_profile.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_risk_profile"))

print(f"  dim_risk_profile: {dim_risk_profile.count()} clienti")

Construire dim_risk_profile...
  dim_risk_profile: 500 clienti


## 5. Verificare — distribuția benzilor de risc

In [0]:
%sql
SELECT 
    risk_band,
    COUNT(*) AS n_customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(risk_score), 1) AS avg_score
FROM banking.gold.dim_risk_profile
GROUP BY risk_band
ORDER BY 
    CASE risk_band 
        WHEN 'LOW' THEN 1 
        WHEN 'MEDIUM' THEN 2 
        WHEN 'HIGH' THEN 3 
        WHEN 'CRITICAL' THEN 4 
    END;

risk_band,n_customers,pct,avg_score
LOW,485,97.0,9.7
MEDIUM,15,3.0,34.0


## 6. Verificare — top 10 clienți cu risc maxim

In [0]:
%sql
SELECT 
    customer_id, segment_code, risk_band, risk_score,
    debt_ratio_score, late_payment_score, kyc_score,
    overdue_loans_count, days_since_last_tx
FROM banking.gold.dim_risk_profile
ORDER BY risk_score DESC
LIMIT 10;

customer_id,segment_code,risk_band,risk_score,debt_ratio_score,late_payment_score,kyc_score,overdue_loans_count,days_since_last_tx
CUST-000394,RETAIL,MEDIUM,43.53,100.0,33.0,30.0,1,1
CUST-000218,RETAIL,MEDIUM,42.69,100.0,33.0,10.0,1,4
CUST-000389,RETAIL,MEDIUM,42.66,100.0,33.0,10.0,1,1
CUST-000128,RETAIL,MEDIUM,39.58,100.0,33.0,10.0,1,2
CUST-000170,RETAIL,MEDIUM,39.5,74.58,33.0,10.0,1,3
CUST-000015,RETAIL,MEDIUM,38.84,96.07,33.0,10.0,1,2
CUST-000006,RETAIL,MEDIUM,32.85,100.0,0.0,10.0,0,3
CUST-000452,RETAIL,MEDIUM,32.52,100.0,0.0,10.0,0,3
CUST-000121,RETAIL,MEDIUM,32.36,100.0,0.0,10.0,0,2
CUST-000486,CORPORATE,MEDIUM,31.49,40.86,33.0,10.0,1,1


## 7. Verificare — distribuții dim_customer_enriched

In [0]:
%sql
-- Distributie pe grupe de varsta
SELECT 
    age_group,
    COUNT(*) AS n_customers,
    ROUND(AVG(total_products), 2) AS avg_products,
    ROUND(AVG(lifetime_value_ron), 0) AS avg_lifetime_value
FROM banking.gold.dim_customer_enriched
GROUP BY age_group
ORDER BY age_group;

age_group,n_customers,avg_products,avg_lifetime_value
18-25,59,2.32,36559.0
26-35,73,2.14,31609.0
36-50,129,2.54,38377.0
51-65,109,2.54,38678.0
66+,130,2.22,34330.0


In [0]:
%sql
-- Distributie pe segmente
SELECT 
    segment_code,
    COUNT(*) AS n_customers,
    ROUND(AVG(total_products), 2) AS avg_products,
    ROUND(AVG(lifetime_value_ron), 0) AS avg_lifetime_value,
    SUM(CASE WHEN has_loan THEN 1 ELSE 0 END) AS with_loan,
    SUM(CASE WHEN has_credit_card THEN 1 ELSE 0 END) AS with_credit_card
FROM banking.gold.dim_customer_enriched
GROUP BY segment_code
ORDER BY avg_lifetime_value DESC;

segment_code,n_customers,avg_products,avg_lifetime_value,with_loan,with_credit_card
RETAIL,392,2.38,36944.0,93,69
PREMIUM,81,2.42,33635.0,15,8
SME,21,2.19,33543.0,4,4
CORPORATE,6,2.17,30511.0,2,1


## 8. Verificare — dim_product

In [0]:
%sql
SELECT category, COUNT(*) AS n_products
FROM banking.gold.dim_product
GROUP BY category
ORDER BY category;

category,n_products
ACCOUNT,4
CARD,6
LOAN,6


## Sumar 03a — Gold Dimensions

Realizat:
- **`dim_product`** — catalog complet cu denumiri comerciale, target_segment, fee_structure
- **`dim_customer_enriched`** — clienți denormalizați + agregate (vârstă, tenure, lifetime value, total produse)
- **`dim_risk_profile`** — 6 indicatori de risc normalizați + scor agregat + benzi LOW/MEDIUM/HIGH/CRITICAL

**Următorul pas:** `03b_gold_aggregates` — KPI-uri zilnice, segmentare RFM, raport P&L lunar.